# AFib CPU results audit and small HRV export

**Open in Colab with Hardware accelerator: None, then Run all.** This notebook mounts your existing Drive and exports only the already-cached HRV features for the 3,891 development windows. It does not train, require a GPU, restore the project source, or read ECG arrays. Do not run the earlier long reference-training cell again for this step.

After export, upload **`hrv_development_features_v1.npz`** back to this conversation. It contains the small arrays needed for feature-quality checks and bounded CPU experiments here. Original model files, cache, and Pi deployment are unaffected.

The two executable cells below define the checked exporter and run it. Progress prints the record currently being read. Drive I/O determines export duration.


## Findings from the uploaded results

Source: `improvement_cpu_v1_results (1).zip` (SHA-256 `0f3f3f487068bd7329366f3ab761df04156a3d5002fb6f8c34b064878704dd5f`). The archive contains ten completed HRV jobs. It contains **no paper_control files**, so it does not include the later three-hour reference run.

All ten prediction files match the frozen master manifest by dataset index, record identifier, start sample, and label. Saved artifact hashes match. Each model covers the same 3,891 development windows exactly once, with record-disjoint validation folds and no excluded outer-fold window present. All saved confusion counts were reproduced. These checks concern exported predictions and metadata; they do not re-verify raw rhythm annotations or HRV extraction.

| Candidate | Mean fold AP | Mean fold AUROC | Mean within-record AUROC | TP | FP | FN | TN |
|---|---:|---:|---:|---:|---:|---:|---:|
| HRV LSTM | 0.16235 | 0.64388 | 0.56185 | 205 | 1,252 | 113 | 2,321 |
| HRV logistic | 0.12410 | 0.60450 | 0.57193 | 221 | 1,614 | 97 | 1,959 |

Counts are pooled **development-validation diagnostics** over 318 positive and 3,573 negative windows. Each fold's threshold was selected on those same validation labels. They therefore give optimistic operating-point estimates and cannot be directly compared with the earlier 4,862-window outer-test confusion counts. Record identifiers still require verification before calling these patient-level splits.

The LSTM has higher AP than logistic regression in all five folds, but it does not have consistently better within-record AUROC. This supports retaining it as a candidate, without establishing a benefit over the ECG+HRV reference.


## Where performance breaks down

| Inner fold | Positive prevalence | LSTM AP | LSTM AUROC | TP | FP | FN | TN |
|---|---:|---:|---:|---:|---:|---:|---:|
| 0 | 6.62% | 0.2642 | 0.8149 | 19 | 25 | 33 | 708 |
| 1 | 7.22% | 0.2286 | 0.7889 | 17 | 25 | 39 | 695 |
| 2 | 8.74% | 0.1016 | 0.5350 | 58 | 452 | 10 | 258 |
| 3 | 12.37% | 0.1387 | 0.5299 | 96 | 642 | 0 | 38 |
| 4 | 5.93% | 0.0786 | 0.5506 | 15 | 108 | 31 | 622 |

Folds 2 and 3 contribute **1,094 of the LSTM's 1,252 false positives (87.4%)**. Both have AUROC near 0.53. In fold 3, the F1-selected threshold predicts positive for 738 of 776 windows: all 96 positives and 642 negatives. This shows why the high recall there is not evidence of strong discrimination.

False negatives are concentrated in folds 0, 1, and 4: 103 of 113. For example, `ltafdb:10` has all 11 positives missed at its fold's threshold despite within-record AUROC 0.8712. Other records have weak ranking too. The evidence points to both ranking and operating-point weaknesses; it does not establish a single cause.

A fixed 50:50 probability average of the two candidates has mean fold AP **0.14607**, below the LSTM alone (**0.16235**). It also loses to the LSTM on each fold. This was a post-hoc development diagnostic, not a new validated model.

The uploaded HRV run took **291.9 seconds overall (4.86 minutes)**. Recorded job durations sum to about **35.1 seconds for all five LSTM jobs** and **2.1 seconds for all five logistic jobs**; the remaining time is outside those job timers. The later ECG reference run has a very different computational cost.


## Next bounded CPU investigation

The predictions ZIP contains model outputs and preprocessing statistics, but no per-window HRV inputs. The next step exports those inputs while preserving the original development folds. No preprocessing statistics are fitted during export; missing HRV values are preserved.

With the exported arrays, the next comparison can test a small nonlinear HRV baseline and features describing changes **inside each ten-minute observation**. Fit any population preprocessing on the inner-training records only. Keep the same five folds and use a fixed small training budget; no random window split or outer-test tuning. The purpose is to test whether the available HRV features contain usable predictive information before committing more time to ECG training.

These are candidate experiments. A gain is not guaranteed, and a small HRV baseline does not automatically replace the paper-aligned hybrid. The incomplete reference run still cannot provide a complete matched comparison.

Method references: [training-only preprocessing](https://scikit-learn.org/stable/common_pitfalls.html), [histogram gradient boosting implementation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html).


## 1. Define the exporter

The exporter checks the exact audited window/label/fold mapping, verifies cache starts and labels, and reads `hrv.npy`, `starts.npy`, `labels.npy`, and metadata for the 53 development records. It rejects inconsistent input before replacing an existing export.

The generated NPZ uses ordinary numeric/string arrays and can be loaded with `allow_pickle=False`. Baseline probabilities are included for comparisons, not as training features. No future-onset timestamp or waveform ECG is exported.


In [ ]:
"""Export the audited development HRV arrays only; does not train or load ECG."""
from pathlib import Path
import csv
import hashlib
import json
import time
import numpy as np

EXPECTED_MANIFEST_SHA256 = 'fb75a772b2011b2318d0b3a82fc117d94ea6f76ce7e109cfcfdfaa2e1f30d050'
EXPECTED_MAPPING_SHA256 = '0a5842b53b7b922fd8827ed77050a5196ace02482fac5e24b35ec8206a160539'
FEATURE_NAMES = ['rmssd_ms', 'sdnn_ms', 'lf_hf_ratio', 'mean_rr_ms', 'pnn50_percent', 'sample_entropy']


def export_hrv_development(run, cache, destination):
    run, cache, destination = Path(run), Path(cache), Path(destination)
    started = time.monotonic()
    experiment = json.loads((run / 'experiment.json').read_text())
    if experiment['config']['manifest_sha256'] != EXPECTED_MANIFEST_SHA256:
        raise ValueError('This is a different manifest from the audited CPU experiment.')
    development = set(map(int, experiment['development_indices']))
    excluded = set(map(int, experiment['excluded_outer_indices']))
    if development & excluded:
        raise ValueError('Development and excluded outer-test windows overlap.')
    models = {}
    input_hashes = {}
    for model in ('hrv_lstm', 'hrv_logistic'):
        by_index = {}
        for fold in range(5):
            path = run / model / f'inner{fold}' / 'validation_predictions.csv'
            input_hashes[f'{model}/inner{fold}'] = hashlib.sha256(path.read_bytes()).hexdigest()
            with path.open(newline='', encoding='utf-8') as handle:
                for row in csv.DictReader(handle):
                    index = int(row['dataset_index'])
                    if index not in development or index in excluded or index in by_index:
                        raise ValueError(f'Duplicate or non-development window: {index}')
                    group = row['subject_id']
                    if '/' in group or '\\' in group or '..' in group:
                        raise ValueError('Unexpected record identifier.')
                    by_index[index] = [group, int(row['start_sample_250hz']), int(row['label']), fold, float(row['probability'])]
        if set(by_index) != development:
            raise ValueError(f'{model}: missing development predictions.')
        models[model] = by_index
    indices = np.asarray(sorted(development), dtype=np.int64)
    rows = models['hrv_lstm']
    canonical = [[int(i), *rows[int(i)][:4]] for i in indices]
    mapping_sha256 = hashlib.sha256(json.dumps(canonical, separators=(',', ':')).encode()).hexdigest()
    if mapping_sha256 != EXPECTED_MAPPING_SHA256:
        raise ValueError('Window identities/labels/folds differ from the audited upload.')
    if any(rows[int(i)][:4] != models['hrv_logistic'][int(i)][:4] for i in indices):
        raise ValueError('The two models do not have matching validation windows.')
    groups = np.asarray([rows[int(i)][0] for i in indices], dtype=str)
    starts = np.asarray([rows[int(i)][1] for i in indices], dtype=np.int64)
    labels = np.asarray([rows[int(i)][2] for i in indices], dtype=np.uint8)
    folds = np.asarray([rows[int(i)][3] for i in indices], dtype=np.int8)
    raw_hrv = np.empty((len(indices), 20, 6), dtype=np.float32)
    records = sorted(set(groups))
    if len(indices) != 3891 or int(labels.sum()) != 318 or len(records) != 53:
        raise ValueError('Unexpected development dataset size or label count.')
    cache_metadata_hash = hashlib.sha256()
    for number, group in enumerate(records, start=1):
        positions = np.flatnonzero(groups == group)
        if len(set(folds[positions])) != 1:
            raise ValueError('A record appears in more than one validation fold.')
        record = cache / group.replace(':', '__')
        print(f'[{number}/{len(records)}] Loading HRV: {group} | elapsed {time.monotonic() - started:.1f}s', flush=True)
        if not (record / 'COMPLETE').is_file():
            raise FileNotFoundError(f'Incomplete cache record: {record}')
        metadata_bytes = (record / 'metadata.json').read_bytes()
        cache_metadata_hash.update(group.encode())
        cache_metadata_hash.update(metadata_bytes)
        cache_starts = np.load(record / 'starts.npy', allow_pickle=False)
        cache_labels = np.load(record / 'labels.npy', allow_pickle=False)
        hrv = np.load(record / 'hrv.npy', mmap_mode='r', allow_pickle=False)
        if cache_starts.ndim != 1 or not np.all(np.diff(cache_starts) > 0):
            raise ValueError(f'Cache starts are not strictly increasing: {group}')
        if cache_labels.shape != cache_starts.shape or hrv.shape != (len(cache_starts), 20, 6):
            raise ValueError(f'Cache shapes disagree: {group}')
        if int(json.loads(metadata_bytes)['window_count']) != len(cache_starts):
            raise ValueError(f'Cache metadata window count disagrees: {group}')
        local = np.searchsorted(cache_starts, starts[positions])
        if (local >= len(cache_starts)).any() or not np.array_equal(cache_starts[local], starts[positions]):
            raise ValueError(f'Cache/window start mismatch: {group}')
        if not np.array_equal(cache_labels[local], labels[positions]):
            raise ValueError(f'Cache/window label mismatch: {group}')
        raw_hrv[positions] = hrv[local]
        del hrv
    if cache_metadata_hash.hexdigest() != experiment['config']['cache_metadata_sha256']:
        raise ValueError('Cache metadata differs from the completed CPU experiment.')
    metadata = {
        'format': 'afib_hrv_development_v1',
        'purpose': 'development-only CPU analysis and bounded model experiments',
        'observation_minutes': 10, 'prediction_horizon_minutes': 20,
        'window_count': len(indices), 'positive_count': int(labels.sum()),
        'record_count': len(records), 'excluded_outer_window_count': len(excluded),
        'outer_test_included': False, 'preprocessing_fitted_during_export': False,
        'mapping_sha256': mapping_sha256, 'input_prediction_sha256': input_hashes,
        'source_config': experiment['config'],
        'caveat': 'These development validation folds were already used for model/checkpoint/threshold selection.',
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix('.npz.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(
            handle, raw_hrv=raw_hrv, labels=labels, dataset_indices=indices,
            subject_ids=groups, start_sample_250hz=starts, validation_folds=folds,
            feature_names=np.asarray(FEATURE_NAMES, dtype=str),
            baseline_lstm_probability=np.asarray([rows[int(i)][4] for i in indices], dtype=np.float64),
            baseline_logistic_probability=np.asarray([models['hrv_logistic'][int(i)][4] for i in indices], dtype=np.float64),
            metadata_json=np.asarray(json.dumps(metadata, sort_keys=True)),
        )
    temporary.replace(destination)
    with np.load(destination, allow_pickle=False) as saved:
        if saved['raw_hrv'].shape != (3891, 20, 6) or not np.array_equal(saved['dataset_indices'], indices):
            raise ValueError('Export verification failed.')
    print(json.dumps({
        'status': 'EXPORTED', 'windows': len(indices), 'positive_windows': int(labels.sum()),
        'shape': list(raw_hrv.shape), 'outer_test_included': False,
        'elapsed_seconds': round(time.monotonic() - started, 1),
        'size_MiB': round(destination.stat().st_size / (1024 ** 2), 2),
        'file': str(destination),
    }, indent=2), flush=True)
    return destination


## 2. Mount Drive and export the small feature file


In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/AFib_Capstone')
RUN = BASE / 'runs/improvement_cpu_v1'
CACHE = BASE / 'cache/v2_chunked_xqrs'
DESTINATION = BASE / 'runs/hrv_development_features_v1.npz'

output = export_hrv_development(RUN, CACHE, DESTINATION)
files.download(str(output))


## Return the result

Upload **`hrv_development_features_v1.npz`** in the conversation. Keep the completed HRV results ZIP and the stopped reference checkpoint in Drive.

This exporter was checked with a synthetic cache using the real audited development mapping: all 3,891 row identities stayed aligned; missing values were preserved; no ECG files were needed; excluded outer windows stayed absent; mismatched cache labels were rejected without overwriting a valid export. Actual HRV feature content can only be assessed once this export is available.
